In [1]:
!pip install opencv-python Pillow transformers gtts ffmpeg-python google-cloud-texttospeech

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 6.0 MB/s eta 0:00:00


In [5]:
cd /kaggle/input/parkvideo

/kaggle/input/parkvideo


In [6]:
ls

'videoplayback (online-video-cutter.com).mp4'


In [8]:
import cv2
import os

def extract_frames(video_path, output_folder, frame_rate=1):
    os.makedirs(output_folder, exist_ok=True)
    
    video = cv2.VideoCapture(video_path)
    fps = video.get(cv2.CAP_PROP_FPS)
    interval = int(fps * frame_rate)  # Capture every second

    count, saved = 0, 0
    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break
        if count % interval == 0:
            frame_path = os.path.join(output_folder, f"frame_{saved}.jpg")
            cv2.imwrite(frame_path, frame)
            saved += 1
        count += 1
    video.release()
    print(f"Extracted {saved} frames.")

extract_frames('videoplayback (online-video-cutter.com).mp4', '/kaggle/working/frames')


Extracted 299 frames.


In [9]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

# Load the model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def generate_caption(image_path):
    image = Image.open(image_path).convert('RGB')
    inputs = processor(image, return_tensors="pt")
    output = model.generate(**inputs)
    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption

# Generate captions for extracted frames
frame_folder = '/kaggle/working/frames'
captions = []
for frame in sorted(os.listdir(frame_folder)):
    frame_path = os.path.join(frame_folder, frame)
    caption = generate_caption(frame_path)
    captions.append(caption)
    print(f"{frame}: {caption}")


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

frame_0.jpg: a street with people walking around it
frame_1.jpg: a city street with people walking around it
frame_10.jpg: a crowd of people walking down a street
frame_100.jpg: a large machine that is making food
frame_101.jpg: a machine that is making food in a kitchen
frame_102.jpg: a large display case filled with lots of food
frame_103.jpg: a man is making pizzas in a kitchen
frame_104.jpg: a man is making pizzas in a restaurant
frame_105.jpg: a man is seen through a window in a building
frame_106.jpg: a man walking down a street in a city
frame_107.jpg: a man is walking down the street in a red shirt
frame_108.jpg: a man walking down a street in a city
frame_109.jpg: a group of people walking down a street
frame_11.jpg: a group of people walking down a street
frame_110.jpg: people walking down a street in the city
frame_111.jpg: people walking down the street in a city
frame_112.jpg: people walking down the street in a city
frame_113.jpg: people sitting on the sidewalk in front o

NameError: name 'git' is not defined